# Cleanview Texas Data Center Scraper
Run the cells from top to bottom. The final cell downloads `texas_datacenter_projects.csv`.

In [ ]:
!pip -q install requests beautifulsoup4 pandas geopy

In [ ]:
"""
cleanview_scraper.py
Colab-ready scraper for publicly visible Texas data-center project pages on Cleanview.

Usage in Colab:
    !pip install requests beautifulsoup4 pandas
    !python cleanview_scraper.py

Output:
    texas_datacenter_projects.csv
"""

import re
import time
from collections import deque
from datetime import date
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.cleanview.co"
START_URL = "https://www.cleanview.co/data-centers/texas"

# Start small for hackathon speed. Increase later if needed.
MAX_DISCOVER = 60
MAX_SCRAPE = 50
REQUEST_DELAY = 1.0

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/120 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(HEADERS)


def get_soup(url):
    r = session.get(url, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def clean_text(text):
    return " ".join(text.split()) if text else None


def extract_number(text):
    if not text:
        return None
    m = re.search(r"([\d,]+(?:\.\d+)?)", text)
    return float(m.group(1).replace(",", "")) if m else None


def discover_project_links():
    soup = get_soup(START_URL)
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if re.match(r"^/data-centers/texas/\d+/", href):
            links.add(urljoin(BASE_URL, href))

    print(f"Seed project links found: {len(links)}")

    queue = deque(links)
    visited = set()
    all_links = set(links)

    while queue and len(visited) < MAX_DISCOVER:
        url = queue.popleft()
        if url in visited:
            continue

        try:
            soup = get_soup(url)
            visited.add(url)

            for a in soup.find_all("a", href=True):
                href = a["href"]
                if re.match(r"^/data-centers/texas/\d+/", href):
                    new_url = urljoin(BASE_URL, href)
                    if new_url not in all_links:
                        all_links.add(new_url)
                        queue.append(new_url)

            print(
                f"\rDiscovered {len(all_links)} project URLs "
                f"after visiting {len(visited)} pages",
                end=""
            )
            time.sleep(REQUEST_DELAY)

        except Exception as e:
            print(f"\nDiscovery failed for {url}: {e}")

    print()
    return sorted(all_links)


def find_value(soup, label):
    matches = soup.find_all(
        string=lambda x: x and x.strip().lower() == label.lower()
    )

    if not matches:
        return None

    start = matches[0].parent

    for nxt in start.find_all_next(limit=8):
        value = clean_text(nxt.get_text(" ", strip=True))
        if value and value.lower() != label.lower():
            return value

    return None


def extract_project(url):
    soup = get_soup(url)

    h1 = soup.find("h1")
    project_name = clean_text(h1.get_text()) if h1 else None

    capacity_text = find_value(soup, "Capacity")
    status = find_value(soup, "Status")
    county = find_value(soup, "County")
    developer = find_value(soup, "Developer")

    description = None
    for p in soup.find_all("p"):
        txt = clean_text(p.get_text(" ", strip=True))
        if project_name and txt and project_name.lower() in txt.lower():
            description = txt
            break

    return {
        "project_name": project_name,
        "developer": developer,
        "county": county,
        "estimated_mw": extract_number(capacity_text),
        "cleanview_status": status,
        "description": description,
        "source_url": url,
    }


def transform_for_streamlit(raw_df):
    stage_mapping = {
        "Planned": "Early Stage",
        "Operating": "Operational",
        "Under Construction": "Construction",
        "Construction": "Construction",
        "Canceled": "Canceled",
        "Cancelled": "Canceled",
    }

    df = raw_df.copy()

    df["stage"] = (
        df["cleanview_status"]
        .map(stage_mapping)
        .fillna("Unknown")
    )

    # Fields to enrich later from geocoding / ERCOT / TCEQ / news.
    df["city"] = ""
    df["latitude"] = None
    df["longitude"] = None
    df["power_type"] = "Unknown"
    df["ercot_status"] = "Not checked"
    df["permit_status"] = "Not checked"
    df["latest_signal"] = (
        "Cleanview status: " + df["cleanview_status"].fillna("Unknown")
    )
    df["source"] = "Cleanview"
    df["last_updated"] = str(date.today())

    cols = [
        "project_name",
        "developer",
        "city",
        "county",
        "latitude",
        "longitude",
        "estimated_mw",
        "stage",
        "power_type",
        "ercot_status",
        "permit_status",
        "latest_signal",
        "source",
        "source_url",
        "last_updated",
    ]

    df = df[cols]
    df = df.dropna(subset=["project_name"])
    df = df.drop_duplicates(subset=["project_name", "developer"])
    return df.reset_index(drop=True)


def main():
    urls = discover_project_links()
    urls = urls[:MAX_SCRAPE]

    print(f"Scraping up to {len(urls)} projects...")

    projects = []

    for i, url in enumerate(urls, start=1):
        try:
            project = extract_project(url)
            projects.append(project)
            print(f"{i}/{len(urls)}  {project['project_name']}")
        except Exception as e:
            print(f"{i}/{len(urls)}  FAILED: {url} -> {e}")

        time.sleep(REQUEST_DELAY)

    raw_df = pd.DataFrame(projects)

    if raw_df.empty:
        raise RuntimeError(
            "No projects were extracted. "
            "The public page structure may have changed."
        )

    final_df = transform_for_streamlit(raw_df)

    output = "texas_datacenter_projects.csv"
    final_df.to_csv(output, index=False)

    print("\nDone.")
    print(f"Projects saved: {len(final_df)}")
    print(f"Output file: {output}")
    print("\nPreview:")
    print(final_df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
# Geocode Texas county centroids for Streamlit map
# Uses OpenStreetMap Nominatim conservatively (cached + 1 request/sec).

import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

CSV_PATH = 'texas_datacenter_projects.csv'
df = pd.read_csv(CSV_PATH)

# Clean county names such as 'Pecos County' -> 'Pecos'
df['county'] = (
    df['county']
    .fillna('')
    .astype(str)
    .str.replace(r'\s+County$', '', regex=True, case=False)
    .str.strip()
)

geolocator = Nominatim(user_agent='texas-datacenter-hackathon-geocoder')
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, swallow_exceptions=True)

# Only geocode each unique county once
county_coords = {}

for county in sorted(df.loc[df['county'] != '', 'county'].unique()):
    query = f'{county} County, Texas, USA'
    location = geocode(query, exactly_one=True, country_codes='us')

    if location:
        county_coords[county] = (location.latitude, location.longitude)
        print(f'OK  {county}: {location.latitude:.4f}, {location.longitude:.4f}')
    else:
        county_coords[county] = (None, None)
        print(f'NOT FOUND  {county}')

# Fill latitude / longitude from county centroid
df['latitude'] = df['county'].map(lambda c: county_coords.get(c, (None, None))[0])
df['longitude'] = df['county'].map(lambda c: county_coords.get(c, (None, None))[1])

# Keep only rows that have coordinates for the map
print('\nRows with coordinates:', df[['latitude', 'longitude']].notna().all(axis=1).sum(), '/', len(df))

# Save updated CSV in the exact filename used by Streamlit
df.to_csv(CSV_PATH, index=False)

display(df[['project_name', 'county', 'latitude', 'longitude']].head(20))


In [ ]:
import pandas as pd
df = pd.read_csv('texas_datacenter_projects.csv')
print('Projects:', len(df))
print('Projects mapped:', df[['latitude','longitude']].notna().all(axis=1).sum())
display(df.head(20))

In [ ]:
from google.colab import files
files.download('texas_datacenter_projects.csv')